# Notebook 6 — Détection d'anomalies sur les métriques — Online Boutique

## Contexte

On applique les mêmes algorithmes que le notebook 03 (Train Ticket)  
sur le deuxième système — Online Boutique.

**Objectif** : vérifier si les résultats se généralisent sur un système différent.

## Différences avec Train Ticket

| Aspect | Train Ticket | Online Boutique |
|--------|-------------|-----------------|
| Services | 41 (Java) | 10 (Go, Python, Node.js) |
| Pannes | 45 (4 types) | 56 (5 types) |
| Dates | Jan 2023 | Août 2022 |

## Algorithmes

| # | Algorithme | Type |
|---|-----------|------|
| 1 | Z-score | Non supervisé |
| 2 | Isolation Forest | Non supervisé |
| 3 | Random Forest | Supervisé |
| 4 | Autoencoder V2 | Deep Learning |

In [1]:
import os, json, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
FIGURES   = PROJET / 'figures/detection_OB'
RESULTS   = PROJET / 'results'
OUTPUT    = PROJET / 'output'

for dossier in [FIGURES]:
    dossier.mkdir(parents=True, exist_ok=True)

DATES_OB  = ['2022-08-22', '2022-08-23']
FAULT_DUR = 3

METRIQUES = [
    'CpuUsageRate(%)',
    'MemoryUsageRate(%)',
    'PodServerLatencyP99(s)',
    'NetworkReceiveBytes',
    'NetworkTransmitBytes',
]

def charger_metriques(date, source):
    metric_dir = source / date / 'metric'
    if not metric_dir.exists():
        return pd.DataFrame()
    dfs = []
    for f in sorted(metric_dir.glob('*_metric.csv')):
        service = f.stem.rsplit('-', 2)[0]
        df = pd.read_csv(f)
        df['service'] = service
        df['datetime'] = pd.to_datetime(
            df['TimeStamp'], unit='s', utc=True
        )
        for col in df.columns:
            if col not in ['Time', 'PodName', 'service', 'datetime']:
                df[col] = pd.to_numeric(
                    df[col].replace('NaN', float('nan')),
                    errors='coerce'
                )
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print("✓ Configuration OK")

I0000 00:00:1783444842.418440   71798 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783444842.418883   71798 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783444842.462800   71798 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✓ Configuration OK


I0000 00:00:1783444844.100127   71798 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783444844.100488   71798 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2. Chargement des données et ground truth

On charge les métriques normales et on génère le ground truth  
pour Online Boutique — 56 pannes sur 5 types.

In [2]:
# Charger les métriques normales
print("Chargement des métriques normales...")
dfs_normal = []
for date in DATES_OB:
    df = charger_metriques(date, NORMAL)
    if not df.empty:
        dfs_normal.append(df)
        print(f"  {date} : {len(df):,} lignes, {df['service'].nunique()} services")

df_normal = pd.concat(dfs_normal, ignore_index=True)

# Charger les métriques avec pannes
print("\nChargement des métriques avec pannes...")
dfs_panne = []
for date in DATES_OB:
    df = charger_metriques(date, ANOMALIES)
    if not df.empty:
        dfs_panne.append(df)
        print(f"  {date} : {len(df):,} lignes")

df_panne = pd.concat(dfs_panne, ignore_index=True)

# Générer le ground truth
rows = []
for date in DATES_OB:
    path = ANOMALIES / date / f'{date}-fault_list.json'
    with open(path) as f:
        data = json.load(f)
    for hour, faults in data.items():
        for fault in faults:
            inject_dt = datetime.strptime(
                fault['inject_time'], '%Y-%m-%d %H:%M:%S'
            )
            service = fault['inject_pod'].rsplit('-', 2)[0]
            for offset in range(FAULT_DUR):
                w_dt = inject_dt + timedelta(minutes=offset)
                rows.append({
                    'date'          : date,
                    'window'        : w_dt.strftime('%H_%M'),
                    'is_anomaly'    : 1,
                    'faulty_service': service,
                    'fault_type'    : fault['inject_type'],
                })

gt_ob = pd.DataFrame(rows)
gt_ob.to_csv(OUTPUT / 'ground_truth_OB.csv', index=False)

print(f"\n✓ Ground truth : {len(gt_ob)} fenêtres")
print(f"  Types : {sorted(gt_ob['fault_type'].unique())}")

Chargement des métriques normales...
  2022-08-22 : 12,090 lignes, 10 services
  2022-08-23 : 7,210 lignes, 10 services

Chargement des métriques avec pannes...
  2022-08-22 : 12,090 lignes
  2022-08-23 : 7,210 lignes

✓ Ground truth : 168 fenêtres
  Types : ['cpu_consumed', 'cpu_contention', 'exception', 'network_delay', 'return']


## 3. Algorithme 1 — Z-score

### Principe

Pour chaque service, on calcule la moyenne et l'écart-type  
de chaque métrique en phase normale. Un point est anormal  
si sa valeur est à plus de 3 écarts-types de la moyenne.

**Pourquoi par service** : chaque service a un comportement différent.  
Le CPU de `frontend` peut normalement être à 24% mais celui de `cart` à 5%.

In [4]:
# Construire la baseline par service
baseline = {}
for service in df_normal['service'].unique():
    df_svc = df_normal[df_normal['service'] == service]
    baseline[service] = {}
    for m in METRIQUES:
        if m in df_svc.columns:
            vals = df_svc[m].dropna()
            baseline[service][m] = {
                'mean': vals.mean(),
                'std' : vals.std()
            }

print(f"✓ Baseline construite pour {len(baseline)} services")

# Aperçu pour un service
print(f"\nExemple — frontend :")
if 'frontend' in baseline:
    for m, stats in baseline['frontend'].items():
        print(f"  {m:<30} moyenne={stats['mean']:.4f} std={stats['std']:.4f}")

✓ Baseline construite pour 10 services

Exemple — frontend :
  CpuUsageRate(%)                moyenne=23.2962 std=7.9952
  MemoryUsageRate(%)             moyenne=16.6117 std=1.9539
  PodServerLatencyP99(s)         moyenne=1.7712 std=3.5192
  NetworkReceiveBytes            moyenne=34072.9999 std=5270.0723
  NetworkTransmitBytes           moyenne=298824.6820 std=58625.5029


### Application du Z-score

Pour chaque point de mesure, on calcule le Z-score de chaque métrique  
par rapport à la baseline du service correspondant.

Une fenêtre est détectée comme anormale si au moins un point  
a un Z-score max > 3.

In [5]:
SEUIL_Z = 3.0

df_panne_z = df_panne.copy()

# Calculer le Z-score pour chaque métrique
for m in METRIQUES:
    z_col = f'z_{m}'
    df_panne_z[z_col] = np.nan
    for service in df_panne_z['service'].unique():
        if service not in baseline or m not in baseline[service]:
            continue
        mean = baseline[service][m]['mean']
        std  = baseline[service][m]['std']
        if std == 0 or np.isnan(std):
            continue
        mask = df_panne_z['service'] == service
        df_panne_z.loc[mask, z_col] = (
            df_panne_z.loc[mask, m] - mean
        ) / std

# Prendre le Z-score max parmi toutes les métriques
z_cols = [f'z_{m}' for m in METRIQUES if f'z_{m}' in df_panne_z.columns]
df_panne_z['z_max']    = df_panne_z[z_cols].abs().max(axis=1)
df_panne_z['anomalie'] = df_panne_z['z_max'] > SEUIL_Z

n_anomalies = df_panne_z['anomalie'].sum()
print(f"Points analysés    : {len(df_panne_z):,}")
print(f"Anomalies détectées: {n_anomalies:,}")
print(f"Taux              : {n_anomalies/len(df_panne_z)*100:.2f}%")

Points analysés    : 19,300
Anomalies détectées: 703
Taux              : 3.64%


### Évaluation par fenêtre

On compare les détections avec le ground truth au niveau  
fenêtre temporelle. Une fenêtre est correctement détectée  
si au moins un point à l'intérieur est marqué comme anormal.

In [6]:
def evaluer_modele(df, gt, dates, col_detection):
    """Évalue un algorithme contre le ground truth par fenêtre."""
    resultats = []
    for date in dates:
        gt_date = gt[gt['date'] == date]
        df_date = df[df['datetime'].dt.date == pd.Timestamp(date).date()]
        for _, row in gt_date.iterrows():
            h, m = map(int, row['window'].split('_'))
            t_debut = pd.Timestamp(f"{date} {h:02d}:{m:02d}:00").tz_localize('UTC')
            t_fin   = t_debut + timedelta(minutes=1)
            mask    = (df_date['datetime'] >= t_debut) & (df_date['datetime'] < t_fin)
            points  = df_date[mask]
            detecte = points[col_detection].any() if not points.empty else False
            resultats.append({
                'date': date, 'window': row['window'],
                'is_anomaly': row['is_anomaly'],
                'faulty_service': row['faulty_service'],
                'fault_type': row['fault_type'],
                'detecte': detecte,
            })
    df_res = pd.DataFrame(resultats)
    VP = ((df_res['is_anomaly'] == 1) & (df_res['detecte'])).sum()
    FP = ((df_res['is_anomaly'] == 0) & (df_res['detecte'])).sum()
    FN = ((df_res['is_anomaly'] == 1) & (~df_res['detecte'])).sum()
    p = VP / (VP + FP) if (VP + FP) > 0 else 0
    r = VP / (VP + FN) if (VP + FN) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return {
        'precision': p, 'rappel': r, 'f1': f,
        'VP': int(VP), 'FP': int(FP), 'FN': int(FN),
    }, df_res

# Évaluer le Z-score
res_z, _ = evaluer_modele(df_panne_z, gt_ob, DATES_OB, 'anomalie')

print("=== Résultats Z-score (Online Boutique) ===")
print(f"  Vrais positifs  (VP) : {res_z['VP']}")
print(f"  Faux positifs   (FP) : {res_z['FP']}")
print(f"  Faux négatifs   (FN) : {res_z['FN']}")
print()
print(f"  Précision : {res_z['precision']*100:.1f}%")
print(f"  Rappel    : {res_z['rappel']*100:.1f}%")
print(f"  F1-score  : {res_z['f1']*100:.1f}%")

=== Résultats Z-score (Online Boutique) ===
  Vrais positifs  (VP) : 96
  Faux positifs   (FP) : 0
  Faux négatifs   (FN) : 72

  Précision : 100.0%
  Rappel    : 57.1%
  F1-score  : 72.7%


## 4. Algorithme 2 — Isolation Forest

### Principe

Isolation Forest par service — même approche que Train Ticket  
avec `contamination=0.10` (optimisé).

L'algorithme construit 100 arbres de décision et considère  
comme anormaux les points faciles à isoler (loin des voisins).

In [8]:
print("Entraînement Isolation Forest par service...")
modeles_if = {}
scalers_if = {}

for service in df_normal['service'].unique():
    df_svc = df_normal[df_normal['service'] == service][METRIQUES].dropna()
    if len(df_svc) < 10:
        continue

    # Normaliser
    scaler = StandardScaler()
    X = scaler.fit_transform(df_svc)

    # Entraîner
    model = IsolationForest(
        n_estimators=100,
        contamination=0.10,
        random_state=42
    )
    model.fit(X)

    modeles_if[service] = model
    scalers_if[service] = scaler

print(f"✓ {len(modeles_if)} modèles entraînés")

Entraînement Isolation Forest par service...
✓ 10 modèles entraînés


### Application de l'Isolation Forest

On applique chaque modèle sur les données de son service.  
IF retourne -1 pour une anomalie et 1 pour un point normal.

In [9]:
df_panne_if = df_panne.copy()
df_panne_if['anomalie_if'] = False

for service in modeles_if.keys():
    mask   = df_panne_if['service'] == service
    df_svc = df_panne_if[mask][METRIQUES].dropna()
    if df_svc.empty:
        continue
    X    = scalers_if[service].transform(df_svc)
    pred = modeles_if[service].predict(X)
    df_panne_if.loc[df_svc.index, 'anomalie_if'] = (pred == -1)

# Évaluer
res_if, _ = evaluer_modele(df_panne_if, gt_ob, DATES_OB, 'anomalie_if')

print("=== Résultats Isolation Forest (Online Boutique) ===")
print(f"  VP : {res_if['VP']}  FP : {res_if['FP']}  FN : {res_if['FN']}")
print(f"  Précision : {res_if['precision']*100:.1f}%")
print(f"  Rappel    : {res_if['rappel']*100:.1f}%")
print(f"  F1-score  : {res_if['f1']*100:.1f}%")

=== Résultats Isolation Forest (Online Boutique) ===
  VP : 132  FP : 0  FN : 36
  Précision : 100.0%
  Rappel    : 78.6%
  F1-score  : 88.0%


## 5. Algorithme 3 — Random Forest (supervisé)

### Principe

Algorithme supervisé — utilise les labels du ground truth  
pour apprendre à distinguer normal et anormal.

Contrairement aux 2 précédents, il voit les exemples anormaux  
pendant l'entraînement.

In [10]:
# Marquer chaque point comme normal ou anormal via les timestamps
rows_gt = []
for date in DATES_OB:
    path = ANOMALIES / date / f'{date}-fault_list.json'
    with open(path) as f:
        data = json.load(f)
    for hour, faults in data.items():
        for fault in faults:
            inject_dt = datetime.strptime(
                fault['inject_time'], '%Y-%m-%d %H:%M:%S'
            )
            for offset in range(FAULT_DUR):
                w_dt = inject_dt + timedelta(minutes=offset)
                rows_gt.append({
                    'debut'  : pd.Timestamp(w_dt).tz_localize('UTC'),
                    'fin'    : pd.Timestamp(
                        w_dt + timedelta(minutes=1)
                    ).tz_localize('UTC'),
                    'service': fault['inject_pod'].rsplit('-', 2)[0],
                })

df_gt_ts = pd.DataFrame(rows_gt)

def est_anomalie(row):
    for _, gt_row in df_gt_ts.iterrows():
        if (row['datetime'] >= gt_row['debut'] and
            row['datetime'] <  gt_row['fin'] and
            row['service']  == gt_row['service']):
            return 1
    return 0

print("Marquage des anomalies (peut prendre 1-2 minutes)...")
df_complet = df_panne.copy()
df_complet['label'] = df_complet.apply(est_anomalie, axis=1)

n_normal = (df_complet['label'] == 0).sum()
n_anomal = (df_complet['label'] == 1).sum()
print(f"\nPoints normaux  : {n_normal:,}")
print(f"Points anormaux : {n_anomal:,}")
print(f"Ratio           : {n_anomal/n_normal*100:.2f}%")

Marquage des anomalies (peut prendre 1-2 minutes)...

Points normaux  : 19,138
Points anormaux : 162
Ratio           : 0.85%


### Entraînement du Random Forest

Avec `class_weight='balanced'` pour compenser le déséquilibre  
(162 anormaux vs 19 138 normaux — ratio 0.85%).

In [11]:
X_rf = df_complet[METRIQUES].dropna()
y_rf = df_complet.loc[X_rf.index, 'label']

# Entraîner
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=10
)
rf.fit(X_rf, y_rf)

# Prédire
y_pred_rf = rf.predict(X_rf)

# Créer la colonne de détection
df_complet_rf = df_complet.loc[X_rf.index].copy()
df_complet_rf['anomalie_rf'] = (y_pred_rf == 1)

# Évaluer
res_rf, _ = evaluer_modele(df_complet_rf, gt_ob, DATES_OB, 'anomalie_rf')

print("=== Résultats Random Forest (Online Boutique) ===")
print(f"  VP : {res_rf['VP']}  FP : {res_rf['FP']}  FN : {res_rf['FN']}")
print(f"  Précision : {res_rf['precision']*100:.1f}%")
print(f"  Rappel    : {res_rf['rappel']*100:.1f}%")
print(f"  F1-score  : {res_rf['f1']*100:.1f}%")

# Feature Importance
print("\n=== Feature Importance ===")
importances = pd.Series(rf.feature_importances_, index=METRIQUES).sort_values(ascending=False)
for feat, imp in importances.items():
    barre = '█' * int(imp * 40)
    print(f"  {feat:<30} {imp:.4f} {barre}")

=== Résultats Random Forest (Online Boutique) ===
  VP : 117  FP : 0  FN : 51
  Précision : 100.0%
  Rappel    : 69.6%
  F1-score  : 82.1%

=== Feature Importance ===
  CpuUsageRate(%)                0.3573 ██████████████
  PodServerLatencyP99(s)         0.1868 ███████
  MemoryUsageRate(%)             0.1728 ██████
  NetworkReceiveBytes            0.1466 █████
  NetworkTransmitBytes           0.1366 █████


## 6. Algorithme 4 — Autoencoder V2 (par fenêtre)

### Principe

Contrairement à l'Autoencoder V1 qui apprend sur les points bruts,  
V2 agrège d'abord les métriques par fenêtre puis apprend  
uniquement sur les 2 fenêtres normales agrégées.

Cette approche a montré F1 = 99.6% sur Train Ticket.

In [12]:
def agreger_metriques_fenetre(date, source, fenetre):
    """Agrège les métriques d'une fenêtre en features."""
    metric_dir = source / date / 'metric'
    if not metric_dir.exists():
        return None
    dfs = []
    for f in sorted(metric_dir.glob('*_metric.csv')):
        df = pd.read_csv(f)
        df['datetime'] = pd.to_datetime(df['TimeStamp'], unit='s', utc=True)
        for col in METRIQUES:
            if col in df.columns:
                df[col] = pd.to_numeric(
                    df[col].replace('NaN', float('nan')), errors='coerce'
                )
        h, m = map(int, fenetre.split('_'))
        t_debut = pd.Timestamp(f"{date} {h:02d}:{m:02d}:00").tz_localize('UTC')
        t_fin   = t_debut + timedelta(minutes=1)
        mask    = (df['datetime'] >= t_debut) & (df['datetime'] < t_fin)
        df_win  = df[mask]
        if not df_win.empty:
            dfs.append(df_win)
    if not dfs:
        return None
    df_all = pd.concat(dfs, ignore_index=True)
    features = {}
    for col in METRIQUES:
        if col in df_all.columns:
            vals = df_all[col].dropna()
            features[f'{col}_mean'] = vals.mean() if len(vals) > 0 else 0
            features[f'{col}_max']  = vals.max() if len(vals) > 0 else 0
            features[f'{col}_std']  = vals.std() if len(vals) > 0 else 0
    return features

# Extraire les features des fenêtres normales
print("Extraction features — fenêtres normales...")
features_norm = []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        fenetre = f.stem.replace('_log', '')
        feat = agreger_metriques_fenetre(date, NORMAL, fenetre)
        if feat:
            feat['label'] = 0
            features_norm.append(feat)

# Extraire les features des fenêtres anormales
print("Extraction features — fenêtres anormales...")
features_anom = []
for _, row in gt_ob.iterrows():
    feat = agreger_metriques_fenetre(row['date'], ANOMALIES, row['window'])
    if feat:
        feat['label'] = 1
        features_anom.append(feat)

df_feat = pd.DataFrame(features_norm + features_anom)
FEATURES_AGG = [c for c in df_feat.columns if c.endswith(('_mean','_max','_std'))]

print(f"\nTotal fenêtres : {len(df_feat)}")
print(f"  Normales  : {(df_feat['label']==0).sum()}")
print(f"  Anormales : {(df_feat['label']==1).sum()}")
print(f"  Features  : {len(FEATURES_AGG)}")

Extraction features — fenêtres normales...
Extraction features — fenêtres anormales...

Total fenêtres : 170
  Normales  : 2
  Anormales : 168
  Features  : 15


### Entraînement de l'Autoencoder V2

Architecture 15 → 8 → 4 → 8 → 15  
Entraînement sur les 2 fenêtres normales agrégées.  
Seuil basé sur le percentile 50 de l'erreur de reconstruction.

In [13]:
X_norm = df_feat[df_feat['label'] == 0][FEATURES_AGG].values
X_all  = df_feat[FEATURES_AGG].values

# Normaliser
scaler_ae = StandardScaler()
X_norm_sc = scaler_ae.fit_transform(X_norm)
X_all_sc  = scaler_ae.transform(X_all)

# Architecture
input_dim = len(FEATURES_AGG)
inputs    = keras.Input(shape=(input_dim,))
enc       = keras.layers.Dense(8, activation='relu')(inputs)
bot       = keras.layers.Dense(4, activation='relu')(enc)
dec       = keras.layers.Dense(8, activation='relu')(bot)
out       = keras.layers.Dense(input_dim, activation='linear')(dec)

ae = keras.Model(inputs, out)
ae.compile(optimizer='adam', loss='mse')

# Entraîner
ae.fit(X_norm_sc, X_norm_sc, epochs=100, batch_size=2, verbose=0)

# Erreur de reconstruction
X_pred      = ae.predict(X_all_sc, verbose=0)
erreurs     = np.mean(np.square(X_all_sc - X_pred), axis=1)
erreurs_n   = np.mean(np.square(
    X_norm_sc - ae.predict(X_norm_sc, verbose=0)
), axis=1)
seuil_ae    = np.percentile(erreurs_n, 50)

# Détection
detecte = erreurs > seuil_ae
vp = ((df_feat['label']==1) & detecte).sum()
fp = ((df_feat['label']==0) & detecte).sum()
fn = ((df_feat['label']==1) & ~detecte).sum()
p  = vp / (vp + fp) if (vp + fp) > 0 else 0
r  = vp / (vp + fn) if (vp + fn) > 0 else 0
f1_ae = 2 * p * r / (p + r) if (p + r) > 0 else 0

res_ae = {
    'precision': p, 'rappel': r, 'f1': f1_ae,
    'VP': int(vp), 'FP': int(fp), 'FN': int(fn),
}

print(f"=== Résultats Autoencoder V2 (Online Boutique) ===")
print(f"  VP : {res_ae['VP']}  FP : {res_ae['FP']}  FN : {res_ae['FN']}")
print(f"  Précision : {res_ae['precision']*100:.1f}%")
print(f"  Rappel    : {res_ae['rappel']*100:.1f}%")
print(f"  F1-score  : {res_ae['f1']*100:.1f}%")

E0000 00:00:1783449361.330372   71798 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


=== Résultats Autoencoder V2 (Online Boutique) ===
  VP : 168  FP : 1  FN : 0
  Précision : 99.4%
  Rappel    : 100.0%
  F1-score  : 99.7%


## 7. Comparaison finale et sauvegarde

Résumé des 4 algorithmes sur les métriques d'Online Boutique.

In [14]:
# Sauvegarder tous les résultats
resultats_ob = pd.DataFrame([
    {'algorithme': 'Z-score', 'systeme': 'Online Boutique',
     'donnees': 'metriques', 'seuil': '3.0',
     'VP': res_z['VP'], 'FP': res_z['FP'], 'FN': res_z['FN'],
     'precision': round(res_z['precision'],4),
     'rappel': round(res_z['rappel'],4),
     'f1': round(res_z['f1'],4)},
    {'algorithme': 'Isolation Forest', 'systeme': 'Online Boutique',
     'donnees': 'metriques', 'seuil': 'contamination=0.10',
     'VP': res_if['VP'], 'FP': res_if['FP'], 'FN': res_if['FN'],
     'precision': round(res_if['precision'],4),
     'rappel': round(res_if['rappel'],4),
     'f1': round(res_if['f1'],4)},
    {'algorithme': 'Random Forest', 'systeme': 'Online Boutique',
     'donnees': 'metriques', 'seuil': 'class_weight=balanced',
     'VP': res_rf['VP'], 'FP': res_rf['FP'], 'FN': res_rf['FN'],
     'precision': round(res_rf['precision'],4),
     'rappel': round(res_rf['rappel'],4),
     'f1': round(res_rf['f1'],4)},
    {'algorithme': 'Autoencoder V2', 'systeme': 'Online Boutique',
     'donnees': 'metriques', 'seuil': 'par_fenetre_P50',
     'VP': res_ae['VP'], 'FP': res_ae['FP'], 'FN': res_ae['FN'],
     'precision': round(res_ae['precision'],4),
     'rappel': round(res_ae['rappel'],4),
     'f1': round(res_ae['f1'],4)},
])

resultats_all = pd.read_csv(RESULTS / 'resultats_detection.csv')
resultats_all = pd.concat([resultats_all, resultats_ob], ignore_index=True)
resultats_all = resultats_all.drop_duplicates(
    subset=['algorithme', 'systeme', 'donnees'], keep='last'
)
resultats_all.to_csv(RESULTS / 'resultats_detection.csv', index=False)

print("✓ Résultats sauvegardés")
print()
ob = resultats_all[
    (resultats_all['systeme'] == 'Online Boutique') &
    (resultats_all['donnees'] == 'metriques')
].sort_values('f1', ascending=False)
print("=== Online Boutique — Métriques ===")
print(ob[['algorithme','f1','VP','FP','FN']].to_string(index=False))

# Comparaison TT vs OB
print("\n=== Comparaison Train Ticket vs Online Boutique ===")
print(f"{'Algorithme':<20} {'TT F1':>10} {'OB F1':>10} {'Diff':>10}")
print("-" * 55)
comparaisons = [
    ('Z-score',           97.7, res_z['f1']*100),
    ('Isolation Forest',  99.6, res_if['f1']*100),
    ('Random Forest',     99.6, res_rf['f1']*100),
    ('Autoencoder V2',    99.6, res_ae['f1']*100),
]
for algo, tt, ob in comparaisons:
    diff = ob - tt
    print(f"{algo:<20} {tt:>9.1f}% {ob:>9.1f}% {diff:>+9.1f}pts")

✓ Résultats sauvegardés

=== Online Boutique — Métriques ===
      algorithme     f1  VP  FP  FN
  Autoencoder V2 0.9970 168   1   0
Isolation Forest 0.8800 132   0  36
   Random Forest 0.8211 117   0  51
         Z-score 0.7273  96   0  72

=== Comparaison Train Ticket vs Online Boutique ===
Algorithme                TT F1      OB F1       Diff
-------------------------------------------------------
Z-score                   97.7%      72.7%     -25.0pts
Isolation Forest          99.6%      88.0%     -11.6pts
Random Forest             99.6%      82.1%     -17.5pts
Autoencoder V2            99.6%      99.7%      +0.1pts
